In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
pd.set_option('display.max_colwidth', 9999999)

# Dataset

In [3]:
df_movies = pd.read_csv('./data/movie_genres.csv')
df_movies

,movie id,year,ave rating,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Horror,Mystery,Romance,Sci-Fi,Thriller
0,6874,2003,3.961832,1,0,0,0,0,1,0,0,0,0,0,0,0,1
1,8798,2004,3.761364,1,0,0,0,0,1,0,1,0,0,0,0,0,1
2,46970,2006,3.250000,1,0,0,0,1,0,0,0,0,0,0,0,0,0
3,48516,2006,4.252336,0,0,0,0,0,1,0,1,0,0,0,0,0,1
4,58559,2008,4.238255,1,0,0,0,0,1,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50879,168250,2017,3.633333,0,0,0,0,0,0,0,0,0,1,0,0,0,0
50880,168250,2017,3.633333,0,0,0,0,0,0,0,0,0,1,0,0,0,0
50881,168250,2017,3.633333,0,0,0,0,0,0,0,0,0,1,0,0,0,0
50882,168250,2017,3.633333,0,0,0,0,0,0,0,0,0,1,0,0,0,0


In [4]:
df_movies['movie id'].value_counts()

movie id
5669     1160
8464     1000
8622      740
4306      680
79132     572
         ... 
69784      10
62155      10
32029      10
55830      10
68793      10
Name: count, Length: 847, dtype: int64

> **There are 50884 rows to match with the viewer dataset and the rating dataset but only 847 movies**

In [5]:
df_users = pd.read_csv('./data/user_preferences.csv')
df_users

,user id,rating count,rating ave,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Horror,Mystery,Romance,Sci-Fi,Thriller
0,2,22,4.00,3.95,4.25,0.00,0.00,4.00,4.12,4.0,4.04,0.00,3.00,4.00,0.00,3.88,3.89
1,2,22,4.00,3.95,4.25,0.00,0.00,4.00,4.12,4.0,4.04,0.00,3.00,4.00,0.00,3.88,3.89
2,2,22,4.00,3.95,4.25,0.00,0.00,4.00,4.12,4.0,4.04,0.00,3.00,4.00,0.00,3.88,3.89
3,2,22,4.00,3.95,4.25,0.00,0.00,4.00,4.12,4.0,4.04,0.00,3.00,4.00,0.00,3.88,3.89
4,2,22,4.00,3.95,4.25,0.00,0.00,4.00,4.12,4.0,4.04,0.00,3.00,4.00,0.00,3.88,3.89
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50879,610,514,3.69,3.55,3.70,3.94,3.88,3.71,3.77,4.0,3.87,3.61,3.44,3.91,3.67,3.61,3.60
50880,610,514,3.69,3.55,3.70,3.94,3.88,3.71,3.77,4.0,3.87,3.61,3.44,3.91,3.67,3.61,3.60
50881,610,514,3.69,3.55,3.70,3.94,3.88,3.71,3.77,4.0,3.87,3.61,3.44,3.91,3.67,3.61,3.60
50882,610,514,3.69,3.55,3.70,3.94,3.88,3.71,3.77,4.0,3.87,3.61,3.44,3.91,3.67,3.61,3.60


In [6]:
df_users['user id'].value_counts()

user id
414    1245
68      967
610     947
249     920
232     861
       ... 
194       1
493       1
3         1
502       1
342       1
Name: count, Length: 397, dtype: int64

> **There are 50884 rows to match with the movie dataset and the rating dataset but only 397 viewers**

In [7]:
df_ratings = pd.read_csv('./data/ratings.csv', header=None)
df_ratings.columns = ['ratings']
df_ratings

,ratings
0,4.0
1,3.5
2,4.0
3,4.0
4,4.5
...,...
50879,5.0
50880,5.0
50881,5.0
50882,5.0


> **The first row of viewer dataset is the viewer who rated the film from the first row of movie dataset 
a point of ... which is the first row of rating dataset**

In [8]:
df_movie_info = pd.read_csv('./data/movie_info.csv')
df_movie_info

,movieId,title,genres
0,4054,Save the Last Dance (2001),Drama|Romance
1,4069,"Wedding Planner, The (2001)",Comedy|Romance
2,4148,Hannibal (2001),Horror|Thriller
3,4149,Saving Silverman (Evil Woman) (2001),Comedy|Romance
4,4153,Down to Earth (2001),Comedy|Fantasy|Romance
...,...,...,...
842,174055,Dunkirk (2017),Action|Drama|Thriller
843,176371,Blade Runner 2049 (2017),Sci-Fi
844,177765,Coco (2017),Adventure|Animation|Children
845,179819,Star Wars: The Last Jedi (2017),Action|Adventure|Fantasy|Sci-Fi


In [9]:
df_movie_info.movieId.nunique()

847

> **847 rows (movies), no duplicates**

# Utility Matrix

In [10]:
movie_id = df_movies['movie id']
user_id = df_users['user id']

df_all = pd.concat([movie_id, user_id, df_ratings], axis=1)
df_all

,movie id,user id,ratings
0,6874,2,4.0
1,8798,2,3.5
2,46970,2,4.0
3,48516,2,4.0
4,58559,2,4.5
...,...,...,...
50879,168250,610,5.0
50880,168250,610,5.0
50881,168250,610,5.0
50882,168250,610,5.0


In [11]:
utility_matrix = df_all.pivot_table(index='movie id', columns='user id', values='ratings')
utility_matrix

user id,2,3,4,7,9,10,12,13,15,16,...,598,599,600,601,603,605,606,607,608,610
movie id,,,,,,,,,,,,,,,,,,,,,
4054,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.5,3.0,3.0,2.0
4069,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2.5,NaN,NaN,NaN,2.5,3.0,NaN,NaN
4148,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,...,NaN,2.5,NaN,NaN,2.0,NaN,2.5,NaN,4.5,3.5
4149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4153,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174055,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,4.0,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN
176371,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.5,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN
177765,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,4.5,NaN,NaN,NaN,NaN,NaN,NaN


# Preprocessing

In [12]:
def scale_data(df_items, df_users, df_ratings):
    """
    Function to scale item, user, and rating data.
    
    Parameters:
    - df_items: DataFrame containing item features
    - df_users: DataFrame containing user features
    - df_ratings: DataFrame containing ratings
    
    Returns:
    - items_scaled: numpy array of scaled item features (first 3 columns removed)
    - users_scaled: numpy array of scaled user features (first 3 columns removed)
    - ratings_scaled: numpy array of ratings scaled to [-1, 1]
    - scalers: dictionary containing the scalers for later reuse
    """
    
    # Scale items
    scaler_item = StandardScaler()
    items_scaled = scaler_item.fit_transform(df_items)
    items_scaled = items_scaled[:, 3:]  # drop first 3 columns
    
    # Scale users
    scaler_user = StandardScaler()
    users_scaled = scaler_user.fit_transform(df_users)
    users_scaled = users_scaled[:, 3:]  # drop first 3 columns
    
    # Scale ratings to range [-1, 1]
    scaler_rating = MinMaxScaler(feature_range=(-1, 1))
    ratings_scaled = scaler_rating.fit_transform(df_ratings)
    
    scalers = {
        "scaler_item": scaler_item,
        "scaler_user": scaler_user,
        "scaler_rating": scaler_rating
    }
    
    return items_scaled, users_scaled, ratings_scaled, scalers

In [13]:
items_scaled, users_scaled, ratings_scaled, scalers = scale_data(
    df_movies, df_users, df_ratings
)

items_scaled.shape, users_scaled.shape, ratings_scaled.shape

((50884, 14), (50884, 14), (50884, 1))

In [14]:
item_train, item_test = train_test_split(items_scaled, train_size=0.70, shuffle=True, random_state=0)
user_train, user_test = train_test_split(users_scaled, train_size=0.70, shuffle=True, random_state=0)
y_train, y_test = train_test_split(ratings_scaled, train_size=0.70, shuffle=True, random_state=0)

item_train.shape, user_train.shape, y_train.shape

((35618, 14), (35618, 14), (35618, 1))

# Training

In [15]:
def build_model(n_user_features, n_item_features, output_units=32, hidden_units=[256, 128], dropout_rate=0.3):
    """
    Build a neural network model for user-item interaction using a two-tower architecture with dropout.
    
    Parameters:
    - n_user_features: int, number of features for user input
    - n_item_features: int, number of features for item input
    - output_units: int, dimension of the embedding/vector output of each tower
    - hidden_units: list of ints, number of units for hidden layers in each tower
    - dropout_rate: float, dropout rate for hidden layers (0.0 = no dropout)
    
    Returns:
    - model: a compiled Keras Model that outputs cosine similarity between user and item embeddings
    """
    
    def build_tower(name):
        tower = keras.Sequential(name=name)
        for units in hidden_units:
            tower.add(keras.layers.Dense(units, activation='relu'))
            if dropout_rate > 0:
                tower.add(keras.layers.Dropout(dropout_rate))
        tower.add(keras.layers.Dense(output_units))
        return tower
    
    # Build towers
    user_tower = build_tower('user_tower')
    item_tower = build_tower('item_tower')
    
    # Input layers
    input_user = keras.layers.Input(shape=(n_user_features,), name='user_input')
    input_item = keras.layers.Input(shape=(n_item_features,), name='item_input')
    
    # Output embeddings
    output_user = user_tower(input_user)
    output_item = item_tower(input_item)
    
    # Dot product (cosine similarity)
    final_output = keras.layers.Dot(axes=1, normalize=True, name='cosine_similarity')([output_item, output_user])
    
    # Build model
    model = keras.Model(inputs=[input_item, input_user], outputs=[final_output, output_item, output_user], name='user_item_model')
    
    return model

In [16]:
model = build_model(
    n_user_features=user_train.shape[1],
    n_item_features=item_train.shape[1],
    output_units=32,
    hidden_units=[256, 128],
    dropout_rate=0.2
)

model.summary()

Model: "user_item_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ item_input (InputLayer)       │ (None, 14)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ user_input (InputLayer)       │ (None, 14)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ item_tower (Sequential)       │ (None, 32)                │          40,864 │ item_input[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ user_tower (Sequential)       │ (None, 32)                │          40,864 │ user_input[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ cosine_similarity (Dot)       │ (None, 1)                 │               0 │ item_tower[0][0],          │
│                               │                           │                 │ user_tower[0][0]           │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 81,728 (319.25 KB)

 Trainable params: 81,728 (319.25 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
EPOCHS = 100
LEARNING_RATE = 2e-3

early_stop = EarlyStopping(monitor='val_loss', patience=5, min_delta=1e-4, restore_best_weights=True)
# reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)

model.compile(
    optimizer=keras.optimizers.Adam(LEARNING_RATE),
    loss={'cosine_similarity': 'mse'}
)

model.fit(
    x=[item_train, user_train],
    y={'cosine_similarity': y_train},
    validation_data=([item_test, user_test], {'cosine_similarity': y_test}),
    epochs=EPOCHS,
    batch_size=128,
    shuffle=True,
    callbacks=[early_stop]
)

Epoch 1/100
279/279 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.1452 - val_loss: 0.1244
Epoch 2/100
279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1225 - val_loss: 0.1173
Epoch 3/100
279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1161 - val_loss: 0.1149
Epoch 4/100
279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1132 - val_loss: 0.1134
Epoch 5/100
279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1122 - val_loss: 0.1105
Epoch 6/100
279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1074 - val_loss: 0.1088
Epoch 7/100
279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1054 - val_loss: 0.1103
Epoch 8/100
279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1046 - val_loss: 0.1079
Epoch 9/100
279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1019 - val_loss: 0.1080
Epoch 10/100
279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1028 - val_loss: 0.1075
Epoch 11/100
279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1011 - val_loss: 0.1064
Epoch 12/100
279/279 ━━━━━━━━━━━━━━━━━━━━

# Add a new User

In [18]:
new_user_id = 5000
new_rating_count = 3
new_rating_ave = 0.0
new_action = 0.0
new_adventure = 5.0
new_animation = 0.0
new_childrens = 0.0
new_comedy = 0.0
new_crime = 0.0
new_documentary = 0.0
new_drama = 0.0
new_fantasy = 5.0
new_horror = 0.0
new_mystery = 0.0
new_romance = 0.0
new_scifi = 0.0
new_thriller = 0.0

new_user = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])
new_user.shape

(1, 17)

> *The new user enjoys movies from the `adventure`, `fantasy` genres. Let's find the top-rated movies for the new user*

In [19]:
# Repeat the array 847 times along axis 0 and 1 times along axis 1 to match with Movie Dataset
new_user_tiled = np.tile(new_user, (len(df_movie_info), 1))
new_user_tiled.shape

(847, 17)

In [20]:
df_movies_deduplicated = df_movies.drop_duplicates(keep='first', ignore_index=True)
df_movies_deduplicated.shape

(847, 17)

# Prediction

In [21]:
def predict_ratings(model, user_vector, df_movies, scalers):
    """
    Predict ratings for a single user across all movies.
    
    Parameters:
    - model: trained Keras model
    - user_vector: 1D array or DataFrame representing the user features
    - df_movies: DataFrame of movie/item features
    - scalers: dictionary containing 'scaler_user', 'scaler_item', 'scaler_rating'
    
    Returns:
    - ratings_predicted: numpy array of predicted ratings in original scale
    """
    
    # Replicate user vector for all movies
    user_repeated = np.tile(user_vector, (len(df_movies), 1))
    
    # Scale user feature
    user_scaled = scalers['scaler_user'].transform(user_repeated)
    user_scaled = user_scaled[:, 3:]

    # Scale item features
    item_scaled = scalers['scaler_item'].transform(df_movies)
    item_scaled = item_scaled[:, 3:]
    
    # Predict ratings
    ratings_scaled, item_embeddings, user_embeddings = model.predict([item_scaled, user_scaled])
    
    # Inverse transform to original rating scale
    ratings_predicted = scalers['scaler_rating'].inverse_transform(ratings_scaled)
    
    return ratings_predicted, item_embeddings, user_embeddings

In [22]:
ratings_pred, item_embeddings, _ = predict_ratings(model, new_user, df_movies_deduplicated, scalers)
ratings_pred.shape

C:\Users\manh\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


(847, 1)

In [23]:
df_my_ratings = pd.concat([pd.Series(ratings_pred.reshape(-1)), df_movies_deduplicated], axis=1)
df_my_ratings = df_my_ratings.rename(columns={0: 'my_ratings_pred'})

In [24]:
df_merged = pd.merge(df_movie_info, df_my_ratings, left_on='movieId', right_on='movie id', how='inner')
df_merged = df_merged[['my_ratings_pred', 'ave rating', 'genres', 'title', 'movie id']]
df_merged.sort_values(by=['my_ratings_pred', 'ave rating'], axis=0, ascending=False, inplace=True)
df_merged.reset_index(drop=True, inplace=True)
df_merged.head(20)

,my_ratings_pred,ave rating,genres,title,movie id
0,4.113607,3.816901,Adventure|Fantasy|Thriller,Harry Potter and the Goblet of Fire (2005),40815
1,3.972121,3.750000,Children|Drama|Mystery,Hugo (2011),90866
2,3.970471,3.862069,Adventure|Drama|Fantasy,Harry Potter and the Order of the Phoenix (2007),54001
3,3.970471,3.954545,Adventure|Drama|Fantasy,"Fall, The (2006)",59387
4,3.970471,3.636364,Adventure|Drama|Fantasy,The Jungle Book (2016),137857
5,3.937402,3.761682,Adventure|Children|Fantasy,Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001),4896
6,3.937402,3.541667,Adventure|Children|Fantasy,"Chronicles of Narnia: Prince Caspian, The (2008)",59501
7,3.937402,3.443548,Adventure|Children|Fantasy,"Chronicles of Narnia: The Lion, the Witch and the Wardrobe, The (2005)",41566
8,3.937402,3.111111,Adventure|Children|Fantasy,"Golden Compass, The (2007)",56171
9,3.904835,4.106061,Adventure|Fantasy,"Lord of the Rings: The Fellowship of the Ring, The (2001)",4993


# Finding Similar Movies

In [25]:
item_embeddings.shape

(847, 32)

In [26]:
def compute_distance_matrix(X, mask_lower_triangle=True):
    """
    Compute the Euclidean distance matrix between all points in X.
    
    Parameters
    ----------
    X : np.ndarray
        Input data matrix of shape (n_movies, n_features). Each row represents a point.
        
    mask_lower_triangle : bool, default=True
        If True, sets the lower triangular part (including the diagonal) of the distance matrix to np.inf.
        This is useful for algorithms that only need the upper triangular distances (e.g., nearest neighbors).
    
    Returns
    -------
    np.ndarray
        A square matrix of shape (n_samples, n_samples), where entry (i, j) contains the Euclidean
        distance between X[i] and X[j]. If mask_lower_triangle=True, the lower triangle is filled with np.inf.
    
    Notes
    -----
    - This implementation is fully vectorized using NumPy, so it is efficient for large datasets.
    - The Euclidean distance is computed using the formula:
    
        ||x_i - x_j||^2 = ||x_i||^2 + ||x_j||^2 - 2 * (x_i · x_j)
        
      Then the square root is taken to get the actual distance.
    """
    
    length_squared = np.sum(np.square(X), axis=1)  # shape of (n_movies, )
    dot_product = np.dot(X, X.T)  # shape of (n_movies, n_movies)

    squared_distance_matrix = length_squared[:, np.newaxis] + length_squared[np.newaxis, :] - 2 * dot_product  # np.newaxis for broadcasting
    distance_matrix = np.sqrt(np.abs(squared_distance_matrix))

    if mask_lower_triangle:
        mask = np.tril(np.ones_like(distance_matrix, dtype=bool), k=0)
        distance_matrix[mask] = np.inf

    return distance_matrix

In [27]:
distance_matrix = compute_distance_matrix(item_embeddings)

df_distance_matrix = pd.DataFrame(distance_matrix, columns=None).round(2)
df_distance_matrix.columns = df_movies_deduplicated['movie id']
df_distance_matrix.index = df_movies_deduplicated['movie id']

df_distance_matrix

movie id,6874,8798,46970,48516,58559,60756,68157,71535,74458,77455,...,31221,36708,37380,46335,7150,51412,85510,93363,111364,5128
movie id,,,,,,,,,,,,,,,,,,,,,
6874,inf,11.48,29.10,27.270000,24.049999,31.270000,24.450001,30.450001,36.150002,37.099998,...,34.580002,32.980000,29.500000,11.480000,31.270000,31.680000,34.490002,33.669998,33.669998,38.730000
8798,inf,inf,22.82,19.620001,15.370000,23.379999,15.600000,26.240000,29.270000,31.530001,...,27.430000,26.379999,25.120001,0.020000,23.379999,24.549999,29.860001,29.160000,29.160000,34.439999
46970,inf,inf,inf,31.420000,23.740000,14.120000,21.250000,16.389999,27.209999,19.780001,...,15.780000,15.420000,21.830000,22.820000,14.120000,14.020000,17.920000,23.990000,23.990000,21.660000
48516,inf,inf,inf,inf,13.890000,28.889999,15.090000,30.660000,26.500000,33.139999,...,33.880001,31.730000,28.809999,19.620001,28.889999,30.260000,38.180000,36.020000,36.020000,39.060001
58559,inf,inf,inf,inf,inf,24.830000,8.950000,24.299999,27.230000,27.180000,...,26.090000,22.549999,22.780001,15.370000,24.830000,21.860001,31.379999,26.559999,26.559999,32.590000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51412,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,15.460000,19.650000,19.650000,17.860001
85510,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,inf,25.840000,25.840000,15.370000
93363,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,inf,inf,inf,inf,inf,inf,0.000000,29.650000


In [28]:
def find_similar_items(df_distance_matrix, df_item_info, k=5):
    rows = []
    
    for i in range(len(df_distance_matrix)):
        row = {}
        
        # Find current item id from index
        item_id = df_distance_matrix.index[i]
        
        # Find title, genres of current item in df_item_info
        item_info = df_item_info[df_item_info['movieId'] == item_id]
        if not item_info.empty:
            row['title'] = item_info['title'].values[0]
            row['genres'] = item_info['genres'].values[0]
        else:
            row['title'] = None
            row['genres'] = None

        # Find similar items
        sorted_distances = df_distance_matrix.loc[item_id].sort_values(ascending=True)
        similar_ids = sorted_distances.head(k).index
        for j, similar_id in enumerate(similar_ids, start=1):
            similar_info = df_item_info[df_item_info['movieId'] == similar_id]
            if not similar_info.empty:
                row[f'similar_title_{j}'] = similar_info['title'].values[0]
                row[f'similar_genres_{j}'] = similar_info['genres'].values[0]
            else:
                row[f'similar_title_{j}'] = None
                row[f'similar_genres_{j}'] = None
        
        rows.append(row)

    return pd.DataFrame(rows)

In [29]:
find_similar_items(df_distance_matrix, df_movie_info, k=3).head(20)

,title,genres,similar_title_1,similar_genres_1,similar_title_2,similar_genres_2,similar_title_3,similar_genres_3
0,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller,"Punisher, The (2004)",Action|Crime|Thriller,"Bourne Supremacy, The (2004)",Action|Crime|Thriller,"Fast and the Furious, The (2001)",Action|Crime|Thriller
1,Collateral (2004),Action|Crime|Drama|Thriller,"Fast & Furious (Fast and the Furious 4, The) (2009)",Action|Crime|Drama|Thriller,Munich (2005),Action|Crime|Drama|Thriller,Taken (2008),Action|Crime|Drama|Thriller
2,Talladega Nights: The Ballad of Ricky Bobby (2006),Action|Comedy,Night at the Museum: Battle of the Smithsonian (2009),Action|Comedy,Shaolin Soccer (Siu lam juk kau) (2001),Action|Comedy,"Other Guys, The (2010)",Action|Comedy
3,"Departed, The (2006)",Crime|Drama|Thriller,Eastern Promises (2007),Crime|Drama|Thriller,Nightcrawler (2014),Crime|Drama|Thriller,Layer Cake (2004),Crime|Drama|Thriller
4,"Dark Knight, The (2008)",Action|Crime|Drama,Swordfish (2001),Action|Crime|Drama,3:10 to Yuma (2007),Action|Crime|Drama,Ip Man (2008),Action|Drama
5,Step Brothers (2008),Comedy,Anger Management (2003),Comedy,American Pie 2 (2001),Comedy,The Intern (2015),Comedy
6,Inglourious Basterds (2009),Action|Drama,We Were Soldiers (2002),Action|Drama,Jarhead (2005),Action|Drama,Black Hawk Down (2001),Action|Drama
7,Zombieland (2009),Action|Comedy|Horror,Kingsman: The Secret Service (2015),Action|Adventure|Comedy|Crime,Spectre (2015),Action|Adventure|Crime,"Dark Knight Rises, The (2012)",Action|Adventure|Crime
8,Shutter Island (2010),Drama|Mystery|Thriller,Prisoners (2013),Drama|Mystery|Thriller,"Da Vinci Code, The (2006)",Drama|Mystery|Thriller,"Village, The (2004)",Drama|Mystery|Thriller
9,Exit Through the Gift Shop (2010),Comedy|Documentary,"Cabin in the Woods, The (2012)",Comedy|Horror|Sci-Fi|Thriller,Kingsman: The Secret Service (2015),Action|Adventure|Comedy|Crime,Idiocracy (2006),Adventure|Comedy|Sci-Fi|Thriller
